# AI-VS3 Health Index - quick start

Computes the plant-level Health Index from the agronomist annotations (Section 4.3) and analyses the
temporal trajectories (Section 5.7). No API keys or GPU are required.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
from aivs3 import tomato_threat_scoring as hi
from aivs3 import temporal

ANNOTATIONS = ROOT / "data" / "Autonomous_Scouting_Dataset.xlsx"
SHEETS = ["Week_01", "Week_02", "Week_03", "Week_04"]
OUT = ROOT / "outputs" / "quickstart"

## 1. Score every plant-session

In [ ]:
images, levels, plants = hi.score_workbook(ANNOTATIONS, SHEETS, source="annotation",
                                           output_file=OUT / "Agronomist_Threat_Scored.xlsx")
plants[["Session", "Plant_Stem_Number", "Health_Index", "Alert_Level",
        "Plant_Reliability_Q", "Plant_Uncertainty_U", "Main_Issue"]].head(10)

## 2. Inspect one plant: image, level and plant scores

In [ ]:
plant, session = "Plant C1-002", "Date_26_Feb_2026"
levels.query("Plant_Stem_Number == @plant and Session == @session")[
    ["Level", "R_Max", "R_Mean", "Disease_Agreement_D", "Severity_Agreement_K",
     "Agreement_Multiplier_G", "Spread_Multiplier_M", "Level_Threat_T", "Dominant_Issue"]]

## 3. Trajectories (Figure 7, Figure 8, Table 5, Table 6)

In [ ]:
wide = temporal.trajectory_table(plants)
temporal.session_statistics(wide).round(3)

In [ ]:
classified = temporal.classify_trajectories(wide)
temporal.trend_distribution(classified)

In [ ]:
classified.loc[["Plant C1-002", "Plant C1-007", "Plant C1-005", "Plant C1-008", "Plant C1-021"],
               list(wide.columns) + ["Delta", "Trend_Label"]].round(3)

In [ ]:
from IPython.display import Image, display

temporal.plot_population_trend(wide, OUT / "fig7_population_trend.png")
temporal.plot_trajectory_heatmap(wide, OUT / "fig8_trajectory_heatmap.png")
display(Image(OUT / "fig7_population_trend.png", width=550))
display(Image(OUT / "fig8_trajectory_heatmap.png"))

## 4. Coefficients used

In [ ]:
hi.scoring_config_table()